# 01 · Bioactivity data curation for TGF-β receptor type I (ALK5)

**Project:** AI-assisted drug discovery for liver fibrosis using PyRMD
**Author:** Hassan Rafiq, Institute of Chemistry, University of the Punjab

### Why ALK5?
Transforming growth factor-β (TGF-β) is the main pro-fibrotic cytokine in the liver. It activates hepatic
stellate cells, which then deposit excess extracellular matrix (collagen) and drive fibrosis. TGF-β signals
through its type I receptor kinase **ALK5 (TGFBR1)**, and ALK5 is the druggable node that small-molecule
inhibitors act on (for example galunisertib). ChEMBL lists this target as **CHEMBL4439** (*Homo sapiens*).

### What this notebook does
1. Retrieves ALK5 bioactivity records (IC50, Ki, Kd, EC50) from ChEMBL
2. Cleans the records and standardizes structures with RDKit (salt stripping, neutralization, canonical SMILES)
3. Merges duplicates by InChIKey and labels each compound **active** or **inactive**
4. Explores the data: potency distribution, drug-likeness, scaffolds, chemical space
5. Exports the training sets for PyRMD and prepares a screening library of approved drugs (drug repurposing)

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import Crippen, Descriptors, Draw, Lipinski, rdFingerprintGenerator, rdMolDescriptors
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem.Scaffolds import MurckoScaffold

RDLogger.DisableLog("rdApp.*")
sns.set_theme(style="whitegrid", context="notebook")

## Settings
Every choice that affects the result is set here, so it can be changed and re-run in one place.

* **Active:** potency ≤ 1 µM (pActivity ≥ 6), a common hit threshold for kinase inhibitors.
* **Inactive:** potency ≥ 10 µM (pActivity ≤ 5).
* Compounds between 1 and 10 µM are a "grey zone" and are left out of training, so the two classes are clearly separated.

In [ ]:
TARGET_CHEMBL_ID = "CHEMBL4439"          # TGF-beta receptor type-1 (ALK5), Homo sapiens
TARGET_LABEL = "tgfbr1"
ACTIVITY_TYPES = ["IC50", "Ki", "Kd", "EC50"]
ACTIVE_CUTOFF_NM = 1_000                 # <= 1 uM  -> active
INACTIVE_CUTOFF_NM = 10_000              # >= 10 uM -> inactive

# "api": download directly from the ChEMBL web services (needs internet)
# "csv": use a file exported from the ChEMBL website (target page -> Activities -> Download CSV)
DATA_SOURCE = "api"
CHEMBL_CSV = "../data/raw/chembl_export.csv"   # only used when DATA_SOURCE = "csv"

# Screening library: "api" = ChEMBL approved small-molecule drugs (max_phase = 4),
# or a path to your own .smi/.csv file with SMILES and an ID
LIBRARY_SOURCE = "api"

RAW, PROCESSED, FIGURES = Path("../data/raw"), Path("../data/processed"), Path("../figures")
for folder in (RAW, PROCESSED, FIGURES):
    folder.mkdir(parents=True, exist_ok=True)

## 1 · Retrieve bioactivity data from ChEMBL

In [ ]:
KEEP = ["molecule_chembl_id", "canonical_smiles", "standard_type", "standard_relation",
        "standard_value", "standard_units", "pchembl_value", "assay_chembl_id", "assay_type",
        "data_validity_comment"]


def fetch_activities_api(target_id, types):
    from chembl_webresource_client.new_client import new_client
    records = new_client.activity.filter(target_chembl_id=target_id, standard_type__in=types).only(KEEP)
    return pd.DataFrame.from_records(list(records))


def load_activities_csv(path):
    """Read a CSV exported from the ChEMBL website and rename columns to the API names."""
    rename = {"Molecule ChEMBL ID": "molecule_chembl_id", "Smiles": "canonical_smiles",
              "Standard Type": "standard_type", "Standard Relation": "standard_relation",
              "Standard Value": "standard_value", "Standard Units": "standard_units",
              "pChEMBL Value": "pchembl_value", "Assay ChEMBL ID": "assay_chembl_id",
              "Assay Type": "assay_type", "Data Validity Comment": "data_validity_comment"}
    df = pd.read_csv(path, sep=None, engine="python").rename(columns=rename)[list(rename.values())]
    # the website export wraps relations in quotes, e.g. "'>'"
    df["standard_relation"] = df["standard_relation"].astype(str).str.strip("'\" ")
    return df[df["standard_type"].isin(ACTIVITY_TYPES)].reset_index(drop=True)


raw = fetch_activities_api(TARGET_CHEMBL_ID, ACTIVITY_TYPES) if DATA_SOURCE == "api" else load_activities_csv(CHEMBL_CSV)
raw = raw.reindex(columns=KEEP)
raw.to_csv(RAW / f"{TARGET_LABEL}_activities_raw.csv", index=False)
print(f"Retrieved {len(raw):,} activity records for {TARGET_CHEMBL_ID}")
raw.head()

## 2 · Clean the records

In [ ]:
df = raw.copy()
df["standard_value"] = pd.to_numeric(df["standard_value"], errors="coerce")
df["data_validity_comment"] = df["data_validity_comment"].replace({"": np.nan, "nan": np.nan, "None": np.nan})
log = {"Raw records": len(df)}

df = df.dropna(subset=["canonical_smiles", "standard_value"])
log["With SMILES and a numeric value"] = len(df)
df = df[(df["standard_units"] == "nM") & (df["standard_value"] > 0)]
log["Reported in nM"] = len(df)
df = df[df["data_validity_comment"].isna()]
log["No ChEMBL data-validity warning"] = len(df)
df = df[df["standard_relation"].isin(["=", "<", "<=", ">", ">="])]
log["Valid relation (=, <, >)"] = len(df)

pd.Series(log, name="records").to_frame()

## 3 · Standardize structures with RDKit
Each SMILES is cleaned up, reduced to its largest fragment (removes salts and solvents) and neutralized.
The InChIKey of the standardized molecule is used to recognise the same compound reported in different papers.

In [ ]:
_chooser = rdMolStandardize.LargestFragmentChooser()
_uncharger = rdMolStandardize.Uncharger()


def standardize(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    try:
        mol = rdMolStandardize.Cleanup(mol)
        mol = _chooser.choose(mol)
        mol = _uncharger.uncharge(mol)
        Chem.SanitizeMol(mol)
    except Exception:
        return None
    return mol


def standardize_table(frame, smiles_col):
    unique = frame[smiles_col].dropna().unique()
    mols = {s: standardize(s) for s in unique}
    std = {s: (Chem.MolToSmiles(m), Chem.MolToInchiKey(m)) if m is not None else (None, None)
           for s, m in mols.items()}
    frame = frame.copy()
    frame["std_smiles"] = frame[smiles_col].map(lambda s: std.get(s, (None, None))[0])
    frame["inchikey"] = frame[smiles_col].map(lambda s: std.get(s, (None, None))[1])
    return frame.dropna(subset=["std_smiles", "inchikey"])


df = standardize_table(df, "canonical_smiles")
print(f"{len(df):,} records with a valid standardized structure, "
      f"covering {df['inchikey'].nunique():,} unique compounds")

## 4 · Label each compound
Each record is labelled first. A censored value such as "> 10 000 nM" still proves the compound is inactive,
and "< 100 nM" still proves it is active. A compound is kept only if its records agree.
Compounds with contradicting records are removed.

In [ ]:
def label_record(row):
    v, rel = row["standard_value"], row["standard_relation"]
    if v <= ACTIVE_CUTOFF_NM and rel in ("=", "<", "<="):
        return "active"
    if v >= INACTIVE_CUTOFF_NM and rel in ("=", ">", ">="):
        return "inactive"
    return "grey"


df["record_label"] = df.apply(label_record, axis=1)


def summarise(group):
    labels = set(group["record_label"]) - {"grey"}
    exact = group.loc[group["standard_relation"] == "=", "standard_value"]
    median_nm = exact.median() if len(exact) else group["standard_value"].median()
    if labels == {"active"}:
        label = "active"
    elif labels == {"inactive"}:
        label = "inactive"
    elif labels == {"active", "inactive"}:
        label = "conflict"
    else:
        label = "grey"
    return pd.Series({"chembl_id": group["molecule_chembl_id"].iloc[0],
                      "smiles": group["std_smiles"].iloc[0],
                      "n_records": len(group),
                      "median_nM": median_nm,
                      "pActivity": 9 - np.log10(median_nm),
                      "label": label})


compounds = df.groupby("inchikey").apply(summarise, include_groups=False).reset_index()
compounds["label"].value_counts().rename("compounds").to_frame()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(data=compounds, x="pActivity", hue="label", bins=40, multiple="stack",
             hue_order=["active", "inactive", "grey", "conflict"],
             palette={"active": "#2a9d8f", "inactive": "#e76f51", "grey": "#bbbbbb", "conflict": "#6c757d"}, ax=ax)
ax.axvline(9 - np.log10(ACTIVE_CUTOFF_NM), ls="--", c="k", lw=1)
ax.axvline(9 - np.log10(INACTIVE_CUTOFF_NM), ls="--", c="k", lw=1)
ax.set(title=f"ALK5 ({TARGET_CHEMBL_ID}) potency distribution", xlabel="pActivity (−log10 M)", ylabel="Compounds")
fig.tight_layout()
fig.savefig(FIGURES / "01_potency_distribution.png", dpi=200)
plt.show()

## 5 · Drug-likeness of actives vs inactives
Lipinski's rule of five and TPSA show whether the actives look like oral drugs.

In [ ]:
train = compounds[compounds["label"].isin(["active", "inactive"])].copy()
train["mol"] = train["smiles"].apply(Chem.MolFromSmiles)


def properties(mol):
    mw, logp = Descriptors.MolWt(mol), Crippen.MolLogP(mol)
    hbd, hba = Lipinski.NumHDonors(mol), Lipinski.NumHAcceptors(mol)
    return pd.Series({"MW": mw, "LogP": logp, "HBD": hbd, "HBA": hba,
                      "TPSA": rdMolDescriptors.CalcTPSA(mol),
                      "RotB": Lipinski.NumRotatableBonds(mol),
                      "Ro5_violations": int(mw > 500) + int(logp > 5) + int(hbd > 5) + int(hba > 10)})


train = pd.concat([train, train["mol"].apply(properties)], axis=1)

fig, axes = plt.subplots(1, 5, figsize=(16, 3.6))
for ax, prop in zip(axes, ["MW", "LogP", "HBD", "HBA", "TPSA"]):
    sns.boxplot(data=train, x="label", y=prop, hue="label", palette={"active": "#2a9d8f", "inactive": "#e76f51"},
                ax=ax, legend=False)
    ax.set(xlabel="", title=prop)
fig.suptitle("Physicochemical profile of the training set", y=1.02)
fig.tight_layout()
fig.savefig(FIGURES / "01_properties.png", dpi=200, bbox_inches="tight")
plt.show()

train.groupby("label")[["MW", "LogP", "HBD", "HBA", "TPSA", "RotB", "Ro5_violations"]].median().round(2)

## 6 · Scaffolds and chemical space
Bemis–Murcko scaffolds show which core structures dominate the actives. A PCA of Morgan fingerprints
(radius 2, 2048 bits) shows whether actives and inactives occupy different regions of chemical space,
which is what a ligand-based model such as PyRMD needs to learn from.

In [ ]:
def scaffold(mol):
    try:
        return Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(mol))
    except Exception:
        return ""


train["scaffold"] = train["mol"].apply(scaffold)
actives = train[train["label"] == "active"]
print(f"{actives['scaffold'].nunique()} unique scaffolds among {len(actives)} actives")
top_scaffolds = actives["scaffold"].value_counts().head(8)
Draw.MolsToGridImage([Chem.MolFromSmiles(s) for s in top_scaffolds.index], molsPerRow=4, subImgSize=(260, 200),
                     legends=[f"n = {n}" for n in top_scaffolds.values])

In [ ]:
morgan = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
fps = [morgan.GetFingerprint(m) for m in train["mol"]]
X = np.array([list(fp) for fp in fps], dtype=np.uint8)
coords = PCA(n_components=2, random_state=0).fit_transform(X)
train["PC1"], train["PC2"] = coords[:, 0], coords[:, 1]

fig, ax = plt.subplots(figsize=(6.5, 5))
sns.scatterplot(data=train, x="PC1", y="PC2", hue="label", s=18, alpha=0.7,
                palette={"active": "#2a9d8f", "inactive": "#e76f51"}, ax=ax)
ax.set(title="Chemical space of ALK5 training compounds (Morgan FP, PCA)")
fig.tight_layout()
fig.savefig(FIGURES / "01_chemical_space_pca.png", dpi=200)
plt.show()

# Diversity: mean nearest-neighbour Tanimoto similarity within each class
for lab in ["active", "inactive"]:
    idx = np.where(train["label"].values == lab)[0]
    sub = [fps[i] for i in idx]
    nn = [max(DataStructs.BulkTanimotoSimilarity(fp, sub[:i] + sub[i + 1:]) or [0]) for i, fp in enumerate(sub)]
    print(f"{lab:>8}: mean nearest-neighbour Tanimoto = {np.mean(nn):.2f}")

In [ ]:
most_potent = actives.sort_values("pActivity", ascending=False).head(12)
Draw.MolsToGridImage(list(most_potent["mol"]), molsPerRow=4, subImgSize=(260, 200),
                     legends=[f"{c}  pAct {p:.1f}" for c, p in zip(most_potent["chembl_id"], most_potent["pActivity"])])

## 7 · Export the PyRMD training sets
PyRMD reads CSV files with a `Smiles` column and a `Title` column.

In [ ]:
for lab in ["active", "inactive"]:
    out = train.loc[train["label"] == lab, ["smiles", "chembl_id"]].rename(columns={"smiles": "Smiles", "chembl_id": "Title"})
    out.to_csv(PROCESSED / f"{TARGET_LABEL}_{lab}s.csv", index=False)
    print(f"Saved {len(out):,} {lab}s -> {PROCESSED / f'{TARGET_LABEL}_{lab}s.csv'}")

train.drop(columns=["mol"]).to_csv(PROCESSED / f"{TARGET_LABEL}_training_set_annotated.csv", index=False)

## 8 · Prepare the screening library (drug repurposing)
Approved drugs already have known safety profiles in humans, so any hit could be repositioned faster than a
new chemical entity. The library is standardized in the same way as the training set.
Drugs that are already in the training set are flagged.

In [ ]:
def fetch_approved_drugs():
    from chembl_webresource_client.new_client import new_client
    records = new_client.molecule.filter(max_phase=4, molecule_type="Small molecule").only(
        ["molecule_chembl_id", "pref_name", "molecule_structures"])
    rows = [{"Title": r["molecule_chembl_id"], "name": r.get("pref_name"),
             "input_smiles": (r.get("molecule_structures") or {}).get("canonical_smiles")} for r in records]
    return pd.DataFrame(rows)


def load_library(path):
    lib = pd.read_csv(path, sep=None, engine="python", header=None if str(path).endswith(".smi") else "infer")
    if str(path).endswith(".smi"):
        lib = lib.iloc[:, :2]
        lib.columns = ["input_smiles", "Title"]
    else:
        smi_col = next(c for c in lib.columns if "smile" in c.lower())
        id_col = next((c for c in lib.columns if c != smi_col), None)
        lib = lib.rename(columns={smi_col: "input_smiles", id_col: "Title"})
    lib["name"] = lib.get("name", "")
    return lib[["Title", "name", "input_smiles"]]


library = fetch_approved_drugs() if LIBRARY_SOURCE == "api" else load_library(LIBRARY_SOURCE)
n_in = len(library)
library = standardize_table(library, "input_smiles").drop_duplicates("inchikey")
library["in_training_set"] = library["inchikey"].isin(set(train["inchikey"]))
library = library.rename(columns={"std_smiles": "Smiles"})
library[["Smiles", "Title"]].to_csv(PROCESSED / "screening_library.csv", index=False)
library[["Title", "name", "Smiles", "inchikey", "in_training_set"]].to_csv(PROCESSED / "screening_library_annotated.csv", index=False)
print(f"Screening library: {n_in:,} input structures -> {len(library):,} unique standardized compounds "
      f"({int(library['in_training_set'].sum())} already in the training set)")

## 9 · Curation summary
These numbers are saved to `data/processed/curation_summary.json` and reused in the results notebook.

In [ ]:
summary = {
    "target": f"TGF-beta receptor type-1 / ALK5 ({TARGET_CHEMBL_ID})",
    "activity_types": ACTIVITY_TYPES,
    "active_cutoff_nM": ACTIVE_CUTOFF_NM,
    "inactive_cutoff_nM": INACTIVE_CUTOFF_NM,
    "raw_records": int(log["Raw records"]),
    "clean_records": int(len(df)),
    "unique_compounds": int(compounds.shape[0]),
    "actives": int((train["label"] == "active").sum()),
    "inactives": int((train["label"] == "inactive").sum()),
    "grey_zone_excluded": int((compounds["label"] == "grey").sum()),
    "conflicting_excluded": int((compounds["label"] == "conflict").sum()),
    "active_scaffolds": int(actives["scaffold"].nunique()),
    "screening_library_size": int(len(library)),
}
(PROCESSED / "curation_summary.json").write_text(json.dumps(summary, indent=2))
pd.Series(summary).to_frame("value")